# FingerBall v2 — Stage 2C：手動ROI・手部マスク・21点骨格動画
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/atstuyuki/FingerBall-Analysis/blob/main/notebooks/FingerBall_v2_Stage2C_ManualROI_HandMask_Landmarks.ipynb)

**今回の短期目標**：リリース前後の指定フレーム区間だけ、手領域のマスクと21点2D骨格を重畳したMP4、ランドマーク座標CSV、ROI/検出品質CSV、2値マスクPNGを出力します。

- フレーム区間・ROIの3キーフレームは **Mac対応ipywidgetsスライダー**で指定。間のROIは線形補間します。
- 骨格は**MediaPipe Hand Landmarker（21点2D）**。WiLoRの未検証2Dカメラ投影を使わず、手の見え方を優先します。
- マスクは**ROIと21点から初期化したGrabCut暫定マスク**で、教師ありhand segmentationではありません。指やボールの正確な境界を保証しません。
- 検出できなかったフレームは **NaN**、マスクは空として残します。存在しない骨格を捏造しません。
- 研究データ・画像・重みはGitHubに入れないでください。Colabによるデータ処理について研究倫理と同意条件を確認してください。

## 0. セットアップとGoogle Drive

In [ ]:
%pip -q install "mediapipe==0.10.35" "opencv-python-headless>=4.8" ipywidgets pandas matplotlib tqdm

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 1. 設定 — 基本的にここだけ変更

In [ ]:
from pathlib import Path
from google.colab import files, output
from IPython.display import display
import numpy as np, pandas as pd, cv2, mediapipe as mp
import ipywidgets as widgets
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
import urllib.request, shutil, json, zipfile
from datetime import datetime
output.enable_custom_widget_manager()

PROJECT_PATH = ''  # 例 '/content/drive/MyDrive/母指回内トラッキング'
VIDEO_PATH = ''    # 空なら手動アップロード
START_FRAME, END_FRAME = 175, 210
ANCHOR_FRAMES = (175, 190, 205)  # 3つのROIキーフレーム（解析区間内）
CROP_SCALE = 2.0     # 手のBOXを拡大してモデルに入力
HAND_MIN_DET = 0.15
HAND_MIN_PRESENCE = 0.2
HAND_MIN_TRACK = 0.2
RUN_GRABCUT = True
RELEASE_ZOOM = 3.0
OUTPUT_DIR_NAME = 'FingerBall_v2_stage2c_results'
TASK_URLS = [
 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/latest/hand_landmarker.task',
 'https://storage.googleapis.com/mediapipe-models/hand_landmarker/hand_landmarker/float16/1/hand_landmarker.task']
def find_project():
    if PROJECT_PATH:
        p=Path(PROJECT_PATH)
        if not p.is_dir():raise FileNotFoundError(p)
        return p
    roots=[Path('/content/drive/MyDrive'),Path('/content/drive/Shareddrives')]
    for root in roots:
        if (root/'母指回内トラッキング').is_dir():return root/'母指回内トラッキング'
        if root.name=='Shareddrives' and root.is_dir():
            for d in root.iterdir():
                if (d/'母指回内トラッキング').is_dir():return d/'母指回内トラッキング'
    raise FileNotFoundError('PROJECT_PATHを指定してください')
project=find_project()
if not VIDEO_PATH:
    uploaded=files.upload()
    if not uploaded:raise RuntimeError('動画が指定されません')
    VIDEO_PATH='/content/'+next(iter(uploaded))
video_original=Path(VIDEO_PATH)
if not video_original.is_file():raise FileNotFoundError(video_original)
cache=Path('/content/fingerball_stage2c');cache.mkdir(exist_ok=True)
if str(video_original).startswith('/content/drive/'):
    video=cache/('input_'+video_original.name)
    if not video.exists() or video.stat().st_size!=video_original.stat().st_size:shutil.copy2(video_original,video)
else:video=video_original
cap=cv2.VideoCapture(str(video))
if not cap.isOpened():raise RuntimeError('動画が開けません')
TOTAL=int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
FPS=float(cap.get(cv2.CAP_PROP_FPS))
W=int(cap.get(cv2.CAP_PROP_FRAME_WIDTH));H=int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
cap.release()
assert 0<=START_FRAME<=END_FRAME<TOTAL
assert len(ANCHOR_FRAMES)>=2 and min(ANCHOR_FRAMES)>=START_FRAME and max(ANCHOR_FRAMES)<=END_FRAME
assert sorted(set(ANCHOR_FRAMES))==list(ANCHOR_FRAMES)
print('video:',video_original.name,'total:',TOTAL,'fps:',FPS,'size:',W,H,'range:',START_FRAME,END_FRAME)

## 2. キーフレームのROIをスライダーで指定

**必ずこのセルの実行後、表示されたフレームを順に選んでBOXを確定してください。** `ANCHOR_FRAMES` で例えば 175,190,205 の3箇所を手動設定します。フレーム間はROIの位置と大きさを線形補間します。後続セルはすべてのBOXが確定しているかチェックします。

In [ ]:
anchor_boxes={}
def frame_at(index):
    cap=cv2.VideoCapture(str(video));cap.set(cv2.CAP_PROP_POS_FRAMES,int(index))
    ok,im=cap.read();cap.release()
    if not ok:raise RuntimeError('Read frame '+str(index))
    return im
frames_cache={i:frame_at(i) for i in ANCHOR_FRAMES}
chooser=widgets.Dropdown(options=list(ANCHOR_FRAMES),description='フレーム')
xr=widgets.IntRangeSlider(value=(int(W*.35),int(W*.65)),min=0,max=W-1,description='X範囲',continuous_update=False,layout=widgets.Layout(width='95%'))
yr=widgets.IntRangeSlider(value=(int(H*.2),int(H*.5)),min=0,max=H-1,description='Y範囲',continuous_update=False,layout=widgets.Layout(width='95%'))
preview=widgets.Image(format='png')
btn=widgets.Button(description='このフレームのBOXを確定',button_style='success',icon='check')
msg=widgets.HTML()
def roi_from_sliders():
    x0,x1=xr.value;y0,y1=yr.value
    if x1-x0<12 or y1-y0<12:return None
    return np.array([x0,y0,x1,y1],dtype=float)
def redraw(*args):
    img=frames_cache[chooser.value].copy()
    b=roi_from_sliders()
    if b is not None:
        x0,y0,x1,y1=map(int,b)
        cv2.rectangle(img,(x0,y0),(x1,y1),(0,255,0),3)
    scale=min(1,960/W)
    if scale<1:img=cv2.resize(img,None,fx=scale,fy=scale)
    _,buf=cv2.imencode('.png',img)
    preview.value=buf.tobytes()
    msg.value='確定済み: '+', '.join(map(str,sorted(anchor_boxes)))
def choose(change):
    if change['new'] in anchor_boxes:
        b=anchor_boxes[change['new']]
        xr.value=(int(b[0]),int(b[2]));yr.value=(int(b[1]),int(b[3]))
    redraw()
def confirm(button):
    b=roi_from_sliders()
    if b is None:
        msg.value='<b>BOXの幅と高さは12px以上必要です</b>'
        return
    anchor_boxes[int(chooser.value)]=b.copy()
    redraw()
    print('確定:',chooser.value,b.astype(int).tolist())
chooser.observe(choose,names='value')
xr.observe(redraw,names='value')
yr.observe(redraw,names='value')
btn.on_click(confirm)
redraw()
display(widgets.VBox([chooser,preview,xr,yr,btn,msg]))
print('すべてのキーフレームでBOXを確定してから次のセルへ進んでください')

## 3. ROIの補間プレビュー・MediaPipeモデルの取得

In [ ]:
if sorted(anchor_boxes)!=list(ANCHOR_FRAMES):
    raise RuntimeError('未設定のROIキーフレーム: '+str(sorted(set(ANCHOR_FRAMES)-set(anchor_boxes))))
keys=np.array(sorted(anchor_boxes),dtype=int)
def interp_box(i):
    if i<=keys[0]:return anchor_boxes[int(keys[0])].copy()
    if i>=keys[-1]:return anchor_boxes[int(keys[-1])].copy()
    k=np.searchsorted(keys,i);a,b=int(keys[k-1]),int(keys[k])
    t=(i-a)/(b-a)
    return (1-t)*anchor_boxes[a]+t*anchor_boxes[b]
fig,axs=plt.subplots(1,len(ANCHOR_FRAMES),figsize=(15,5))
for ax,i in zip(np.ravel(axs),ANCHOR_FRAMES):
    img=frames_cache[i].copy();q=np.rint(interp_box(i)).astype(int)
    cv2.rectangle(img,tuple(q[:2]),tuple(q[2:]),(0,255,0),3)
    ax.imshow(cv2.cvtColor(img,cv2.COLOR_BGR2RGB));ax.set_title(f'Frame {i}');ax.axis('off')
plt.show()
task_path=cache/'hand_landmarker.task'
if task_path.exists() and task_path.stat().st_size<1000000:task_path.unlink()
if not task_path.exists():
    errors=[]
    for url in TASK_URLS:
        tmp=cache/'hand_download.tmp'
        try:
            req=urllib.request.Request(url,headers={'User-Agent':'Mozilla/5.0'})
            with urllib.request.urlopen(req,timeout=90) as response,open(tmp,'wb') as out:shutil.copyfileobj(response,out)
            if tmp.stat().st_size<1000000:raise RuntimeError('Model file too small')
            tmp.replace(task_path);break
        except Exception as e:
            errors.append(str(e));tmp.unlink(missing_ok=True)
    if not task_path.exists():raise RuntimeError('モデル取得失敗: '+repr(errors))
print('HandLandmarker model:',task_path.stat().st_size,'bytes')